# Figure 5: Gi* hotspots and bivariate Moran's I (2016-19)

- **Paper:** Fig 5
- **Original notebook:** `34_hotspot_analysis.ipynb`
- **Reads:**
  - `outputs/school_related_pre.pkl`
  - `data/raw/geographic_boundaries/ca_counties/Ca_County_shp`
- **Writes:**
  - `outputs/plots/corrplot.png`
  - `outputs/plots/hotspotmap.png`


In [ ]:
# Paths: edit config.py at the repo root, not this cell
import sys
from pathlib import Path
_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config.py").exists())
sys.path.insert(0, str(_root))
from config import DATA_DIR, DERIVED_DIR, OUTPUT_DIR, FIG_DIR

In [ ]:
# setup

import os
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import random
from sklearn.model_selection import train_test_split
import statsmodels.api as sm
import xgboost as xgb
#from sklearn.metrics import aic


from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score




In [ ]:
# pip install rioxarray  # see requirements.txt
# pip install pysal  # see requirements.txt
# pip install contextily  # see requirements.txt
import libpysal

from matplotlib.patches import Patch
# pip install matplotlib_scalebar  # see requirements.txt
from matplotlib_scalebar.scalebar import ScaleBar


import rioxarray  # Surface data manipulation
import xarray  # Surface data manipulation
from pysal.explore import esda  # Exploratory Spatial analytics
from pysal.lib import weights  # Spatial weights
import contextily

In [ ]:
mergedf = pd.read_pickle(f'{OUTPUT_DIR}/school_related_pre.pkl')
mergedf['pct_passed_math'] = pd.to_numeric(mergedf['pct_passed_math'],errors='coerce')
mergedf['college_going_rate'] = pd.to_numeric(mergedf['college_going_rate'],errors='coerce')

schools =mergedf.groupby('SchoolCode').agg({
    'CountyName' :'first',
    'SchoolName': 'first',
    'DistrictName':'first',
    'Latitude': 'first',
    'Longitude':'first',
    'pct_passed_math':'mean',
    'college_going_rate':'mean',
    'pct_frpm':'mean',
    'AP_Count':'mean'
}, skipna=True).reset_index()
#tract_sum = schoolsin.groupby('TRACTCE').sum().reset_index()

schools_pre = gpd.GeoDataFrame(schools,geometry=gpd.points_from_xy(schools['Longitude'], schools['Latitude']), crs='EPSG:4326')



In [ ]:
sp_fi = pd.read_pickle(f'{OUTPUT_DIR}/school_related_pre.pkl')

sp_fi['pct_passed_math'] = pd.to_numeric(sp_fi['pct_passed_math'],errors='coerce')
sp_fi['college_going_rate'] = pd.to_numeric(sp_fi['college_going_rate'],errors='coerce')

ca_counties = gpd.read_file(f"{DATA_DIR}/geographic_boundaries/ca_counties/Ca_County_shp")

schools_p =mergedf.groupby('SchoolCode').agg({
    'CountyName' :'first',
    'SchoolName': 'first',
    'DistrictName':'first',
    'Latitude': 'first',
    'Longitude':'first',
    'pct_passed_math':'mean',
    'college_going_rate':'mean',
    'ChronicAbsenteeismRate':'mean',
    'pct_frpm':'mean',
    'AP_Count':'mean'
}, skipna=True).reset_index()
#tract_sum = schoolsin.groupby('TRACTCE').sum().reset_index()
schools_p = schools_p.dropna(subset=['Latitude', 'Longitude'])

schools_post = gpd.GeoDataFrame(schools_p,geometry=gpd.points_from_xy(schools_p['Longitude'], schools_p['Latitude']), crs='EPSG:4326')

schools_post = schools_post.to_crs(ca_counties.crs)  # Or vice versa


In [ ]:
xy = schools_p[['Latitude','Longitude','college_going_rate']]
xy.dropna(inplace=True)

coordinates = list(zip(xy['Longitude'], xy['Latitude']))
# w = libpysal.weights.DistanceBand(coordinates,threshold=.1)
w = weights.KNN(coordinates, k=6)  # Ensure every point has at least 5 neighbors
y = np.array(xy['college_going_rate'])
go_local = esda.getisord.G_Local(y,w,transform='B')

go_local.Zs

xy_loc =  gpd.GeoDataFrame(xy,geometry=gpd.points_from_xy(xy['Longitude'], xy['Latitude']), crs='EPSG:4326')

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import Patch
import geopandas as gpd
import pandas as pd
import mapclassify
import matplotlib as mpl

plt.rc('axes.spines', **{'bottom': False, 'left': False, 'right': False, 'top': False})
fig = plt.figure(figsize=(14, 10))

gs = fig.add_gridspec(2, 3, width_ratios=[1, 1, 0.05], height_ratios=[1, 1])

if ca_counties.crs != xy_loc.crs:
    ca_counties = ca_counties.to_crs(xy_loc.crs)


ax1 = fig.add_subplot(gs[0, 0])

ec = "0.8"
sig = go_local.p_sim < 0.05

# basemap
ca_counties.plot(ax=ax1, color='gainsboro', edgecolor='black', linewidth=0.5, alpha=0.5)

# non-significant clusters
ns = xy_loc.loc[sig == False, "geometry"]
ns.plot(ax=ax1, color="grey", edgecolor='white', markersize=10,linewidth=0.1)

# HH clusters
hh = xy_loc.loc[(go_local.Zs > 0) & (sig == True), "geometry"]
hh.plot(ax=ax1, color="red", edgecolor='white',markersize=10, linewidth=0.1)

# LL clusters
ll = xy_loc.loc[(go_local.Zs < 0) & (sig == True), "geometry"]
ll.plot(ax=ax1, color="blue", edgecolor='white', markersize=10,linewidth=0.1)

ax1.set_title("Gi* for College Going Rate", size=12, loc='left')
ax1.set_axis_off()

legend_elements = [
    Patch(facecolor='grey', edgecolor='none', label='Not Significant'),
    Patch(facecolor='red', edgecolor='none', label='High-High Cluster'),
    Patch(facecolor='blue', edgecolor='none', label='Low-Low Cluster')
]
ax1.legend(handles=legend_elements, loc="upper right")#, bbox_to_anchor=(0.8, 0.95))

gdf = gpd.GeoDataFrame(schools_p, geometry=gpd.points_from_xy(schools_p['Longitude'], schools_p['Latitude']), crs='EPSG:4326')
cols = ['college_going_rate', 'ChronicAbsenteeismRate', 'pct_frpm', 'AP_Count', 'pct_passed_math']
gdf[cols] = gdf[cols].apply(pd.to_numeric, errors='coerce')

n_classes = 5
for col in cols:
    classifier = mapclassify.NaturalBreaks(gdf[col].dropna(), k=n_classes)
    gdf[col] = classifier.find_bin(gdf[col])

ax2 = fig.add_subplot(gs[0, 1])
ax3 = fig.add_subplot(gs[1, 0])
ax4 = fig.add_subplot(gs[1, 1])

ca_counties = ca_counties.to_crs(gdf.crs)

cmap = mpl.cm.viridis

# Plot each feature map
for ax, col, title in zip(
    [ax2, ax3, ax4],
    ['college_going_rate', 'ChronicAbsenteeismRate', 'pct_frpm'],
    ['College Going Rate', 'Chronic Absenteeism', 'FRPM']
):
    ca_counties.plot(ax=ax, color='darkgrey', edgecolor='white', linewidth=0.35)
    gdf.plot(column=col, ax=ax, markersize=10, cmap=cmap, legend=False, alpha=0.75,
            edgecolor='k', linewidth=0.2)
    ax.set_axis_off()
    ax.set_title(title, loc='left')

cax = cax = fig.add_axes([0.95, 0.3, 0.02, 0.4])

norm = mpl.colors.Normalize(vmin=0, vmax=n_classes-1)
sm = mpl.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cbar = fig.colorbar(sm, cax=cax, orientation='vertical')
cbar.set_ticks(range(n_classes))
cbar.set_ticklabels([f'Class {i+1}' for i in range(n_classes)])
cbar.set_label('Jenks Natural Breaks Classes')

plt.suptitle('Spatial Distribution of Key Features', y=0.95, fontsize=14, fontweight='bold')


# plt.tight_layout()
plt.subplots_adjust(  wspace=-0.65, hspace=-0.1)
plt.show()

In [ ]:
from shapely.geometry import Point

ca_counties = gpd.read_file(f"{DATA_DIR}/geographic_boundaries/ca_counties/Ca_County_shp")

if ca_counties.crs != xy_loc.crs:
    ca_counties = ca_counties.to_crs(xy_loc.crs)

fig, ax = plt.subplots(1, 1, figsize=(6, 8))

ec = "0.8"

sig = go_local.p_sim < 0.05
# basemap
ca_counties.plot(ax=ax, color='gainsboro', edgecolor='black', linewidth=0.5,alpha = 0.5)

# non-significant clusters
ns = xy_loc.loc[sig == False, "geometry"]
ns.plot(ax=ax, color="#93ABBFFF", edgecolor='white', linewidth=0.1)

# HH clusters
hh = xy_loc.loc[(go_local.Zs > 0) & (sig == True), "geometry"]
hh.plot(ax=ax, color="#DD4124FF", edgecolor='white', linewidth=0.1)

#  LL clusters
ll = xy_loc.loc[(go_local.Zs < 0) & (sig == True), "geometry"]
ll.plot(ax=ax, color="#035AA6FF", edgecolor='white', linewidth=0.1)

coordinates = [
    (34.0549, -118.2426),  # Los Angeles
    (37.7749, -122.4194),  # San Francisco
    (32.7157, -117.1611),  # San Diego
    (38.5781, -121.4944)   # Sacramento
]

points = gpd.GeoDataFrame(
    geometry=[Point(lon, lat) for lat, lon in coordinates],
    crs="EPSG:4326"
)

if points.crs != ca_counties.crs:
    points = points.to_crs(ca_counties.crs)

# Plot the points as small black dots
points.plot(ax=ax, color='black', markersize=2, label='Cities')



st = "*" if go_local.star else ""


# ax.set_title(f"(a) Gi* for College Going Rate", size=15)


ax.set_axis_off()

legend_elements = [
    Patch(facecolor='#93ABBFFF', edgecolor='none', label='Not Significant'),
    Patch(facecolor='#DD4124FF', edgecolor='none', label='Hotspot Cluster'),
    Patch(facecolor='#035AA6FF', edgecolor='none', label='Coldspot Cluster')
]

ax.legend(handles=legend_elements,loc= "upper right", fontsize=16)
# ax.add_artist(ScaleBar(dx=1,units="m"))

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/plots/hotspotmap.png',dpi=600)

plt.show()


In [ ]:


ca_counties = gpd.read_file(f"{DATA_DIR}/geographic_boundaries/ca_counties/Ca_County_shp")

if ca_counties.crs != xy_loc.crs:
    ca_counties = ca_counties.to_crs(xy_loc.crs)

fig, ax = plt.subplots(1, 2, figsize=(14, 8))
ax = ax.flatten()

ec = "0.8"

sig = go_local.p_sim < 0.05
# basemap
ca_counties.plot(ax=ax[0], color='gainsboro', edgecolor='black', linewidth=0.5,alpha = 0.5)

# non-significant clusters
ns = xy_loc.loc[sig == False, "geometry"]
ns.plot(ax=ax[0], color="grey", edgecolor='white', linewidth=0.1)

# HH clusters
hh = xy_loc.loc[(go_local.Zs > 0) & (sig == True), "geometry"]
hh.plot(ax=ax[0], color="red", edgecolor='white', linewidth=0.1)

#  LL clusters
ll = xy_loc.loc[(go_local.Zs < 0) & (sig == True), "geometry"]
ll.plot(ax=ax[0], color="blue", edgecolor='white', linewidth=0.1)



st = "*" if go_local.star else ""


ax[0].set_title(f"(a) Gi* for College Going Rate", size=15,y=1.02)


ax[0].set_axis_off()

legend_elements = [
    Patch(facecolor='grey', edgecolor='none', label='Not Significant'),
    Patch(facecolor='red', edgecolor='none', label='High-High Cluster'),
    Patch(facecolor='blue', edgecolor='none', label='Low-Low Cluster')
]

ax[0].legend(handles=legend_elements,loc= "upper right", bbox_to_anchor=(0.8, 0.95))
# ax.add_artist(ScaleBar(dx=1,units="m"))


# correlation matrix data
metrics = ['College Going Rate', 'Chronic Absenteeism', 'FRPM', 'AP Count', 'SBA Math Proficiency']
correlations = np.array([
    [1.000, -0.248, -0.393, 0.351, 0.385],
    [-0.248, 1.000, 0.212, -0.179, -0.259],
    [-0.393, 0.212, 1.000, -0.192, -0.478],
    [0.351, -0.179, -0.192, 1.000, 0.258],
    [0.385, -0.259, -0.478, 0.258, 1.000]
])
matrix = np.triu(correlations)
# plt.figure(figsize=(10, 8))

sns.heatmap(correlations,ax = ax[1],
            xticklabels=metrics,
            yticklabels=metrics,
            cmap='RdBu',mask = matrix,
            vmin=-1,
            vmax=1,
            center=0, annot_kws={'size': 13},
            annot=True,
            fmt='.3f',
            square=True,
            cbar_kws={'label': "Bivariate Moran's I",'shrink': 0.8})

ax[1].set_xticks(range(len(metrics)))
ax[1].set_xticklabels(metrics, rotation=45, ha='right', fontsize=13)

ax[1].set_yticks(range(len(metrics)))
ax[1].set_yticklabels(metrics, rotation=45, fontsize=13, va='center_baseline')
ax[1].set_title("(b) Bivariate Moran's I in key School-Level Variables", fontsize=15,y=1.02)

plt.show()

In [ ]:


metrics = ['College\nGoing Rate', 'Chronic\nAbsenteeism', 'FRPM', 'AP Count', 'SBA Math\nProficiency']
correlations = np.array([
    [1.000, -0.248, -0.393, 0.351, 0.385],
    [-0.248, 1.000, 0.212, -0.179, -0.259],
    [-0.393, 0.212, 1.000, -0.192, -0.478],
    [0.351, -0.179, -0.192, 1.000, 0.258],
    [0.385, -0.259, -0.478, 0.258, 1.000]
])

new_order = [1, 2, 3, 4, 0]  # New order of indices
metrics_reordered = [metrics[i] for i in new_order]
correlations_reordered = correlations[new_order, :][:, new_order]

matrix = np.triu(correlations_reordered)

# plot
plt.figure(figsize=(10, 8))
hm = sns.heatmap(
    correlations_reordered,
    xticklabels=metrics_reordered,
    yticklabels=metrics_reordered,
    cmap='RdBu',
    mask=matrix,
    vmin=-1,
    vmax=1,
    center=0,
    annot=True,
    fmt='.2f',
    annot_kws={'size': 16},
    square=True,
    cbar_kws={'pad': -0.02}
)

# colorbar
cbar = hm.collections[0].colorbar
cbar.ax.tick_params(labelsize=14)

# axis labels
plt.xticks( fontsize=16)
plt.yticks( fontsize=16)

plt.savefig(f'{OUTPUT_DIR}/plots/corrplot.png',dpi=600)
plt.tight_layout()
plt.show()